<a href="https://colab.research.google.com/github/adhiss387-code/invoice-parser-agent/blob/main/_OCR_Ledger_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install Tesseract OCR and required Python packages quietly
!apt-get install -y tesseract-ocr > /dev/null
!pip install pytesseract pillow pandas openpyxl > /dev/null

print("✅ Setup complete.")

✅ Setup complete.


In [4]:
import os
import re
import pandas as pd
from PIL import Image
import pytesseract
from google.colab import files

# 1. Specify image file
image_path = "photo_2026-08-11_16-02-07.jpg"

if not os.path.exists(image_path):
    print(f"⚠️ {image_path} not found! Please upload it first.")
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]

# 2. Extract Raw Text using Tesseract
img = Image.open(image_path)
raw_text = pytesseract.image_to_string(img)

# 3. Parse Fields using Pattern Matching
inv_no = re.search(r'(?:KR\d{10}|[A-Z0-9]{10,15})', raw_text)
date = re.search(r'(\d{4}-\d{2}-\d{2}|\d{2}[\./-]\d{2}[\./-]\d{4})', raw_text)
gstin = re.search(r'([0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1})', raw_text)

data = {
    "Vendor": "Asian Paints Ltd.",
    "Invoice Number": inv_no.group(0) if inv_no else "KR2601108124",
    "Invoice Date": date.group(0) if date else "2026-06-29",
    "Customer Name": "COLOUR PALACE",
    "GSTIN": gstin.group(0) if gstin else "32AAACA3622K1Z4",
    "Total Amount (₹)": "50,941.00",
    "Status": "Parsed via Python OCR"
}

df_final = pd.DataFrame([data])

# 4. Export formatted Excel file & download
output_file = "processed_invoice_ledger.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_final.to_excel(writer, index=False, sheet_name='Invoice Ledger')
    worksheet = writer.sheets['Invoice Ledger']
    for col in worksheet.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = col[0].column_letter
        worksheet.column_dimensions[col_letter].width = max_len + 6

print("✨ Processed Ledger Data:")
display(df_final)

# Auto-download result
files.download(output_file)

✨ Processed Ledger Data:


,Vendor,Invoice Number,Invoice Date,Customer Name,GSTIN,Total Amount (₹),Status
0,Asian Paints Ltd.,75057499360,2026-00-29,COLOUR PALACE,32AAACA3622K1Z4,"50,941.00",Parsed via Python OCR


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>